# LUXOR9 AI Engine — Google Colab GPU
Run ComfyUI with a free T4 GPU and connect it to your local video pipeline.

**How it works:**
1. This notebook starts ComfyUI with a free T4 GPU
2. Exposes the ComfyUI API via Cloudflare Tunnel
3. Your local LUXOR9 video pipeline sends AI generation requests here
4. Remotion + TTS still run locally (they don't need GPU)

**Cost: $0 — Free T4 GPU from Google Colab**

---
## Step 1: Install ComfyUI

In [ ]:
# Clone ComfyUI
!git clone https://github.com/comfyanonymous/ComfyUI.git /content/ComfyUI
!cd /content/ComfyUI && pip install -q -r requirements.txt 2>&1 | tail -3
print('✅ ComfyUI installed')

---
## Step 2: Download Models

In [ ]:
# Download ComfyUI-compatible models for the T4 GPU
# Using known working checkpoint files (not diffusers format)
import os, requests
from tqdm import tqdm

checkpoint_dir = '/content/ComfyUI/models/checkpoints'
vae_dir = '/content/ComfyUI/models/vae'
os.makedirs(checkpoint_dir, exist_ok=True)
os.makedirs(vae_dir, exist_ok=True)

def download(url, dest):
    if os.path.exists(dest):
        print(f'  ✅ {os.path.basename(dest)} already exists')
        return
    print(f'  Downloading {os.path.basename(dest)}...')
    r = requests.get(url, stream=True)
    total = int(r.headers.get('content-length', 0))
    with open(dest, 'wb') as f:
        for chunk in tqdm(r.iter_content(8192), total=total//8192, unit='KB'):
            f.write(chunk)

# DreamShaper 8 — fast, great quality, 2GB, works on T4
download(
    'https://civitai.com/api/download/models/128713?type=Model&format=SafeTensor',
    os.path.join(checkpoint_dir, 'dreamshaper_8.safetensors')
)

# SDXL VAE
download(
    'https://huggingface.co/madebyollin/sdxl-vae-fp16-fix/resolve/main/sdxl_vae.safetensors',
    os.path.join(vae_dir, 'sdxl_vae.safetensors')
)

# Optional: SDXL Base (6.9GB, takes longer)
# from huggingface_hub import hf_hub_download
# hf_hub_download('stabilityai/stable-diffusion-xl-base-1.0', 'sd_xl_base_1.0.safetensors', local_dir=checkpoint_dir)

print('✅ Models ready for T4 GPU')

---
## Step 3: Install cloudflared for API tunnel

In [ ]:
# Install cloudflared to expose ComfyUI API
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared
print('✅ cloudflared installed')

---
## Step 4: Start ComfyUI + Tunnel

In [ ]:
import subprocess
import threading
import time
import requests
import json

# Start ComfyUI in background
def start_comfyui():
    subprocess.run([
        'python', '/content/ComfyUI/main.py',
        '--listen', '127.0.0.1',
        '--port', '8188',
        '--force-fp16',
        '--highvram'
    ], cwd='/content/ComfyUI')

thread = threading.Thread(target=start_comfyui, daemon=True)
thread.start()

# Wait for ComfyUI to start
print('Starting ComfyUI on T4 GPU...')
for i in range(60):
    try:
        r = requests.get('http://127.0.0.1:8188/system_stats', timeout=2)
        if r.ok:
            print('✅ ComfyUI running on http://127.0.0.1:8188')
            break
    except:
        pass
    time.sleep(2)
else:
    print('❌ ComfyUI failed to start')

# Start Cloudflare Tunnel
tunnel = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'http://127.0.0.1:8188'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
)

# Extract the tunnel URL
comfyui_url = None
for line in iter(tunnel.stdout.readline, ''):
    print(line.strip())
    if 'trycloudflare.com' in line:
        comfyui_url = line.split('//')[1].strip() if '//' in line else line.strip()
        break

if comfyui_url:
    print(f'\n\n🎯 COMFYUI API URL: {comfyui_url}')
    print(f'   Configure your local pipeline:')
    print(f'   export COMFYUI_HOST={comfyui_url}')
    print(f'   export COMFYUI_PORT=443')
    print(f'   export COMFYUI_PROTOCOL=https')
else:
    print('❌ Failed to get tunnel URL')

---
## Step 5: Configure Local Pipeline

On your local machine (WSL2), run:

```bash
# Point the pipeline to this Colab ComfyUI instance
export COMFYUI_HOST=your-tunnel-url.trycloudflare.com
export COMFYUI_PORT=443

# Test the connection
curl https://$COMFYUI_HOST/system_stats

# Run the pipeline (uses Colab GPU for AI gen, local for TTS + Remotion)
npx tsx src/free-demo.ts
```

---
## Step 6: Keep Alive

Colab disconnects after 90 min of inactivity. Use this cell to keep it alive:

```python
# Click the browser console and paste:
function ClickConnect(){
  console.log("Keep alive");
  document.querySelector("colab-connect-button").click()
}
setInterval(ClickConnect, 60000)
```

---
## Architecture

```
Google Colab (Free T4 GPU)
  └── ComfyUI → AI image/video gen
        └── Cloudflare Tunnel → Public API URL
              │
              ▼ (internet)
Your Local WSL2 Machine
  ├── LUXOR9 Pipeline → Calls Colab ComfyUI for AI gen
  ├── gTTS → Voiceover (no GPU needed)
  └── Remotion → Final video composition (no GPU needed)
```